In [1]:
# # Install Pytorch & other libraries
# %pip install "torch==2.4.1" tensorboard 
# %pip install flash-attn "setuptools<71.0.0" scikit-learn 
 
# # Install Hugging Face libraries
# %pip install  --upgrade \
#   "datasets==3.1.0" \
#   "accelerate==1.2.1" \
#   "hf-transfer==0.1.8"
#   #"transformers==4.47.1" \
 
# # ModernBERT is not yet available in an official release, so we need to install it from github
# %pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from datasets import load_dataset, concatenate_datasets
 
# Dataset id from huggingface.co/dataset
dataset_id = "youralien/feedback_qesconv_16wayclassification"
 
# Load raw dataset
raw_dataset = load_dataset(dataset_id, split="train") # happens to be called train

print(f"Raw dataset size: {len(raw_dataset)}")

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw dataset size: 8179


In [4]:
split_dataset = raw_dataset.train_test_split(test_size=0.1)
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")
split_dataset['train'][0]

Train dataset size: 7361
Test dataset size: 818


{'conv_index': 358,
 'helper_index': 2,
 'input': ["Seeker: I'm okay, just a little stressed",
  'Helper: is there anything specifically that is stressing you out the most right now?',
  "Seeker: I'm just having a hard time dealing with hurtful things people have said to me. I feel no self-worth.",
  "Helper: I'm really sorry to hear that you're feeling this way. It must be really difficult for you. Can you tell me more about these situations and how they made you feel?"],
 'Reflections-goodareas': 0,
 'Validation-goodareas': 1,
 'Empathy-goodareas': 1,
 'Questions-goodareas': 1,
 'Suggestions-goodareas': 0,
 'Self-disclosure-goodareas': 0,
 'Structure-goodareas': 0,
 'Professionalism-goodareas': 0,
 'Reflections-badareas': 0,
 'Validation-badareas': 0,
 'Empathy-badareas': 0,
 'Questions-badareas': 0,
 'Suggestions-badareas': 0,
 'Self-disclosure-badareas': 0,
 'Structure-badareas': 0,
 'Professionalism-badareas': 0}

In [5]:
split_dataset['train'][0]['input'][-1]

"Helper: I'm really sorry to hear that you're feeling this way. It must be really difficult for you. Can you tell me more about these situations and how they made you feel?"

In [6]:
split_dataset['train'][0]['input'][-3:-1]

['Helper: is there anything specifically that is stressing you out the most right now?',
 "Seeker: I'm just having a hard time dealing with hurtful things people have said to me. I feel no self-worth."]

In [7]:
# Downsample once before training
majority_samples = split_dataset['train'].filter(lambda example: example['Reflections-goodareas'] == 0)
minority_samples = split_dataset['train'].filter(lambda example: example['Reflections-goodareas'] == 1)
downsampled_majority = majority_samples.shuffle(seed=42).select(range(len(majority_samples) // 3))
balanced_dataset = concatenate_datasets([downsampled_majority, minority_samples]).shuffle(seed=42)

Filter: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7361/7361 [00:00<00:00, 24749.87 examples/s]


In [8]:
balanced_dataset

Dataset({
    features: ['conv_index', 'helper_index', 'input', 'Reflections-goodareas', 'Validation-goodareas', 'Empathy-goodareas', 'Questions-goodareas', 'Suggestions-goodareas', 'Self-disclosure-goodareas', 'Structure-goodareas', 'Professionalism-goodareas', 'Reflections-badareas', 'Validation-badareas', 'Empathy-badareas', 'Questions-badareas', 'Suggestions-badareas', 'Self-disclosure-badareas', 'Structure-badareas', 'Professionalism-badareas'],
    num_rows: 2945
})

## Exploring Hyperparameter Sweeps with WanDB
Link: https://wandb.ai/matt24/vit-snacks-sweeps/reports/Hyperparameter-Search-for-HuggingFace-Transformer-Models--VmlldzoyMTUxNTg0

In [9]:
import wandb
wandb.login()


%env WANDB_PROJECT=Llama3_1_8B_SkillClassifier
# false, checkpoint 
%env WANDB_LOG_MODEL=false

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ryanlouie2021 (ryanlouie2021-stanford-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


env: WANDB_PROJECT=Llama3_1_8B_SkillClassifier
env: WANDB_LOG_MODEL=false


### Actual Sweep with CBL

In [10]:
# # method
# # https://wandb.ai/wandb_fc/articles/reports/What-Is-Bayesian-Hyperparameter-Optimization-With-Tutorial---Vmlldzo1NDQyNzcw
# sweep_config = {
#     'method': 'bayes',
#     'metric': {
#          'name': 'eval/f1', # important to use 'eval/f1' since this is the specific name
#          'goal': 'maximize'  
#     }
# }

# # hyperparameters
# parameters_dict = {
#     'epochs': {
#         'values': [2, 4] # this has a relationship with the linear learning rate schedule. Some folks tried 20!! Our experience is that many epochs is slow, and tends to overfit.
#     },
#     'batch_size': {
#         'values': [8, 32, 64] # 128 wont fit into 24GB GPU memory
#     },
#     'warmup_ratio': {
#         'values': [0.0, 0.1] # 0.0 was HF default that worked well before; 0.06 is used in BERT, 0.1 was used in another paper
#         # 'value': 0.1 # following 10% and then a linear decay; https://openreview.net/pdf?id=nzpLWnVAyah
#     },
#     'learning_rate': {
#         'distribution': 'log_uniform_values',
#         'min': 1e-5,
#         'max': 1e-3
#     },
#     # 'learning_rate': {
#     #     'values': [1e-5, 3e-5, 5e-5, 8e-5] # these values were taken from the ModernBERT hyperparameter sweep of GLUE + some we discovered.
#     # },
#     'weight_decay': {
#         # 0.1, and 0.01 has been used in original RoBERTa GLUE, 0.2 was found to be actually good in some runs? 
#         # 'values': [1e-6, 5e-6, 8e-6, 1e-5] # these smaller values were taken from the ModernBERT hyperparameter sweep of GLUE
#         'values': [0.0, 0.01, 0.1, 0.2]
#         # 'value': 0.0 
#     },
#     'beta': {    
#         'values': [0.3, 0.6, 0.9, 0.99] # Since Beta ranges from [0, 1), we select several parameters that were found to be best in the original paper, also, selecting a few that don't use class balanced loss as much. 0.99 is the inverse loss.
#     },
#     'context_size': {
#         'values': [1, 5, None] # some skills, like Reflections, depend wholy on what was said previously, making it easier to learn. 10 was based on Rose's work. None is full available context
#     }
# }

# sweep_config['parameters'] = parameters_dict


### Sweep just to reproduce Reflections

In [11]:
# https://wandb.ai/wandb_fc/articles/reports/What-Is-Bayesian-Hyperparameter-Optimization-With-Tutorial---Vmlldzo1NDQyNzcw
sweep_config = {
    'method': 'bayes',
    'metric': {
         'name': 'eval/f1', # important to use 'eval/f1' since this is the specific name
         'goal': 'maximize'  
    }
}

# hyperparameters
parameters_dict = {
    'epochs': {
        'values': [4, 10, 20] # this has a relationship with the linear learning rate schedule. Some folks tried 20!! Our experience is that many epochs is slow, and tends to overfit.
    },
    'batch_size': {
        'values': [8, 16] # 128 wont fit into 24GB GPU memory
    },
    # 'warmup_ratio': {
    #     'values': [0.0, 0.1] # following 10% and then a linear decay; https://openreview.net/pdf?id=nzpLWnVAyah
    # },
    'learning_rate': {
        'distribution': 'log_uniform_values',
        'min': 1e-6,
        'max': 1e-5
    },
    # 'learning_rate': {
    #     'values': [1e-5, 3e-5, 5e-5, 8e-5] # these values were taken from the ModernBERT hyperparameter sweep of GLUE + some we discovered.
    # },
    'weight_decay': {
        # 0.1, and 0.01 has been used in original RoBERTa GLUE, 0.2 was found to be actually good in some runs? 
        # 'values': [1e-6, 5e-6, 8e-6, 1e-5] # these smaller values were taken from the ModernBERT hyperparameter sweep of GLUE
        # 'values': [0.0, 0.1, 0.2]
        'value': 0.0 
    },
    # 'beta': {    
    #     'value': 0.999 # Since Beta ranges from [0, 1), we select several parameters that were found to be best in the original paper, also, selecting a few that don't use class balanced loss as much. 0.99 is the inverse loss.
    # },
    'context_size': {
        'value': 1 # some skills, like Reflections, depend wholy on what was said previously, making it easier to learn. 10 was based on Rose's work. None is full available context
    },
    'downsampling_factor': {
        'values': [4, 8]
    },
    'lora_alpha': {
        'value': 16
    },
    'lora_dropout': {
        'values': [0, 0.05]
    },
    'lora_r': {
        'values': [8, 64]
    }
}

sweep_config['parameters'] = parameters_dict


In [12]:
import random
import torch
from torch.utils.data import Sampler, DataLoader

class ImbalancedDatasetSampler(Sampler):
    def __init__(self, dataset, indices=None, downsampling_factor=3):
        # Indices of all samples
        self.indices = list(range(len(dataset))) if indices is None else indices
        
        # Extract labels from the dataset
        self.labels = [dataset[i]['labels'] for i in self.indices]
        self.downsampling_factor = downsampling_factor
        
        # Store reference to the dataset
        self.dataset = dataset
        
    def __iter__(self):
        # Find indices of each class
        majority_indices = [i for i, label in zip(self.indices, self.labels) if label == 0]
        minority_indices = [i for i, label in zip(self.indices, self.labels) if label == 1]
        
        # Randomly select majority samples for this epoch
        random.shuffle(majority_indices)
        downsampled_majority = majority_indices[:len(majority_indices) // self.downsampling_factor]
        
        # Combine with all minority samples
        indices = downsampled_majority + minority_indices
        random.shuffle(indices)
        return iter(indices)
    
    def __len__(self):
        # The length is the number of samples that will be sampled
        labels = torch.tensor(self.labels)
        majority_count = (labels == 0).sum().item() // self.downsampling_factor
        minority_count = (labels == 1).sum().item()
        return majority_count + minority_count

def get_custom_dataloader(dataset, tokenizer, batch_size, downsampling_factor=3):
    # Create the sampler
    sampler = ImbalancedDatasetSampler(dataset, downsampling_factor=downsampling_factor)
    
    # Create data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    # Create the dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=data_collator
    )
    
    return dataloader

In [13]:
# %pip install -U bitsandbytes
# # %pip install -U accelerate
# %pip install -U peft
# %pip install -U trl

In [14]:
# %pip install -U pip


In [15]:
from transformers import AutoModelForSequenceClassification, AutoModelForCausalLM
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments
from transformers import BitsAndBytesConfig
import torch
import gc

import bitsandbytes as bnb
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

import evaluate
import numpy as np

access_token = os.environ.get("HF_TOKEN")

def compute_metrics_fn(eval_preds):
    metrics = dict()
    
    accuracy_metric = evaluate.load('accuracy')
    precision_metric = evaluate.load('precision')
    recall_metric = evaluate.load('recall')
    f1_metric = evaluate.load('f1')
    
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    preds = np.argmax(logits, axis=-1)  
    
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(f1_metric.compute(predictions=preds, references=labels, average='binary'))

    # Print some predictions
    print(f"Some predictions: {preds[:10]}")
    
    return metrics


def get_class_weight(beta, n):
    """
    Compute class-balanced weight:
    alpha = (1 - beta) / (1 - beta^n) where n is the number of samples for the class
    
    Args:
        beta: Hyperparameter for class-balanced loss (typically between 0.9 and 0.999)
        n: Number of samples for a particular class
    
    Returns:
        The weight for the class
    """
    return (1 - beta) / (1 - beta**n)


def compute_class_balanced_loss(outputs, labels, num_items_in_batch, class_balanced_loss):
    """
    Compute class-balanced loss using the provided configuration
    
    Args:
        outputs: Model outputs containing 'logits'
        labels: Ground truth labels
        class_balanced_loss: Dictionary containing 'beta', 'n_0', and 'n_1' parameters
            - beta: Hyperparameter for class-balanced loss
            - n_0: Number of samples for class 0
            - n_1: Number of samples for class 1
    
    Returns:
        Computed loss value
    """
    logits = outputs['logits']
    
    # Compute class weights using the provided beta and class sample counts
    weights = torch.tensor([
        get_class_weight(class_balanced_loss['beta'], class_balanced_loss['n_0']),
        get_class_weight(class_balanced_loss['beta'], class_balanced_loss['n_1'])
    ])
    
    # Normalize weights
    weights = weights / weights.sum()
    
    # Move weights to the same device as logits
    weights = weights.to(device=logits.device)
    
    # Create loss function with computed weights
    criterion = torch.nn.CrossEntropyLoss(weight=weights)
    
    # Compute loss
    loss = criterion(logits, labels)
    
    return loss

def prepare_input_text(example, context_size=1):
    """
    [-6] Seeker: 
    [-5] Helper:
    [-4] Seeker: 
    [-3] Helper:
    [-2] Seeker: 
    [-1] Helper: Response to classify
    """
    # Convert the last two items of input list to a single text
    response_to_classify = example['input'][-1]
    if context_size is None:
        context = "\n".join(example['input'][:-1])
    else:
        context_start_idx = -1 - context_size
        context = "\n".join(example['input'][context_start_idx:-1])
    return {
        'text': f"{context}[SEP]{response_to_classify}",
        **{k:v for k,v in example.items() if k != 'input'}  # Keep other fields
    }

# def prepare_tokenized_binary_classification_dataset(dataset, which_class):
#     """
#     e.g., which_dataset = "Question-goodareas"
#     """
#     # Apply the preprocessing
#     dataset = dataset.map(prepare_input_text)
#     print(dataset['train'][0])
    
#     SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
#     goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
#     badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
#     cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
#     cols_to_remove.extend(goodareas_to_ignore)
#     cols_to_remove.extend(badareas_to_ignore)
#     if which_class in dataset["train"].features.keys():
#         dataset =  dataset.rename_column(which_class, "labels") # to match Trainer
#     tokenized_dataset = dataset.map(lambda batch: tokenizer(batch['text'], truncation=True), batched=True, remove_columns=cols_to_remove)
#     return tokenized_dataset
    
def cleanup(things_to_delete: list | None = None):
    if things_to_delete is not None:
        for thing in things_to_delete:
            if thing is not None:
                del thing

    gc.collect()
    torch.cuda.empty_cache()


def train_model(config, dataset, which_class):

    # Replace your ModernBERT model loading with Llama 3.1
    base_model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=access_token)
    tokenizer.pad_token_id = tokenizer.eos_token_id

    with wandb.init(config=config):
        # set sweep configuration
        config = wandb.config

        def prepare_input_text_fn(example):
            return prepare_input_text(example, context_size=config.context_size)
        
        dataset = dataset.map(prepare_input_text_fn)
        print(dataset['train'][0])
        
        SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
        goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
        badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
        cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
        cols_to_remove.extend(goodareas_to_ignore)
        cols_to_remove.extend(badareas_to_ignore)
        if which_class in dataset["train"].features.keys():
            dataset = dataset.rename_column(which_class, "labels") # to match Trainer
         
        def generate_prompt(example):
            """
            Convert your classification data to instruction format for Llama 3.1
            """
            # Extract the components from your data structure
            context = "\n".join(example['input'][:-1]) if 'input' in example else ""
            response_to_classify = example['input'][-1] if 'input' in example else example['text']
            label = "selected" if example.get('labels', 0) == 1 else "not selected"
            
            return f"""Classify the following response as either "selected" or "not selected" for {which_class}.
Context: {context}
Response: {response_to_classify}
Label: {label}""".strip()
        
        # Apply the prompt formatting to your dataset
        formatted_dataset = dataset.map(
            lambda examples: {"text": generate_prompt(examples)},
            remove_columns=[col for col in dataset["train"].column_names if col != "text"]
        )
        
        # Tokenize the formatted dataset
        def preprocess_function(example):
            return tokenizer(example["text"], truncation=True, max_length=512)
        
        tokenized_dataset = formatted_dataset.map(preprocess_function, batched=True)
        print(tokenized_dataset['train'][0])
        
        # # Create custom dataloader for training
        # train_dataloader = get_custom_dataloader(
        #     tokenized_dataset["train"], 
        #     tokenizer, 
        #     config.batch_size,
        #     downsampling_factor=config.downsampling_factor
        # )

        peft_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            lora_dropout=config.lora_dropout,
            target_modules=["q_proj", "v_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        
        # Quantization configuration for efficient training
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=False,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        
        model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            device_map="auto",
            # torch_dtype="float16",
            quantization_config=bnb_config,
            token=access_token,
        )
        
        # model.config.use_cache = False
        # model.config.pretraining_tp = 1
        
        # Define training args

        training_args = SFTConfig(output_dir=f"llama-3.1-8B-{which_class}-classifier", packing=True)

        # training_args = TrainingArguments(
        #     output_dir=f"llama-3.1-8B-{which_class}-classifier"
        #     num_train_epochs=config.epochs,                      
        #     per_device_train_batch_size=config.batch_size,  # Adjust based on your GPU memory
        #     per_device_eval_batch_size=config.batch_size,
        #     gradient_accumulation_steps=8,            
        #     gradient_checkpointing=True,              
        #     optim="paged_adamw_32bit",
        #     logging_steps=50,                         
        #     learning_rate=config.learning_rate,                       
        #     weight_decay=config.weight_decay,
        #     fp16=True,
        #     bf16=False,
        #     max_grad_norm=0.3,                        
        #     warmup_ratio=0.03,                        
        #     group_by_length=False,
        #     lr_scheduler_type="cosine",               
        #     report_to="wandb",                  
        #     eval_strategy="steps",
        #     eval_steps=0.2
        # )
        
        # training_args = TrainingArguments(
        #     output_dir= f"ModernBERT-{which_class}-classifier-sweeps",
        #     per_device_train_batch_size=config.batch_size,
        #     per_device_eval_batch_size=16,
        #     learning_rate=config.learning_rate,
        #     warmup_ratio=config.warmup_ratio, 
        #     num_train_epochs=config.epochs,
        #     weight_decay=config.weight_decay,
        #     bf16=True, # bfloat16 training 
        #     optim="adamw_torch_fused", # improved optimizer 
        #     # logging & evaluation strategies
        #     logging_strategy="epoch",
        #     logging_steps=100,
        #     eval_strategy="epoch",
        #     save_strategy="no", # epoch, no
        #     # save_total_limit=1, # needs to be commented out if save_strategy=no
        #     # load_best_model_at_end=True, # needs to be commented out if save_strategy=no
        #     # use_mps_device=True, # mps device is a mac thing
        #     # push to hub parameters
        #     report_to="wandb",
        #     # push_to_hub=True,
        #     # hub_strategy="every_save",
        #     # hub_token=HfFolder.get_token(),
        #     use_legacy_prediction_loop=True,  # Important for custom dataloader
        # )

        ##########
        # Option 3: Downsample + Upweight Majority
        ##########
        def compute_class_balanced_loss_fn(outputs, labels, num_items_in_batch):
            logits = outputs['logits']
            
            # Define weights based on your downsampling factor
            # If you downsampled by factor of 3, the weight for majority class should be 3
            weights = torch.tensor([config.downsampling_factor, 1.0], device=logits.device)  # [majority_weight, minority_weight]
            
            criterion = torch.nn.CrossEntropyLoss(weight=weights)
            loss = criterion(logits, labels)
            return loss
        
        # hf_data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        
        
        # # Create a Trainer instance
        # trainer = Trainer(
        #     model=model,
        #     args=training_args,
        #     train_dataset=tokenized_dataset["train"],
        #     eval_dataset=tokenized_dataset["test"],
        #     processing_class=tokenizer,
        #     data_collator=hf_data_collator,
        #     compute_metrics=compute_metrics_fn,
        #     compute_loss_func=compute_class_balanced_loss_fn, # custom class balanced loss
        # )

        trainer = SFTTrainer(
            model=model,
            train_dataset=tokenized_dataset['train'],
            eval_dataset=tokenized_dataset['test'],
            peft_config=peft_config,
            tokenizer=tokenizer,
            args=training_args,
        )

        
        # # Override the default dataloader
        # trainer.get_train_dataloader = lambda: train_dataloader
        try:
            trainer.train()
            cleanup(things_to_delete=[model, trainer, tokenizer, tokenized_dataset])
        except:
            cleanup(things_to_delete=[model, trainer, tokenizer, tokenized_dataset])


In [16]:
def run_sweep(which_class):
    sweep_id = wandb.sweep(sweep_config, project=f'llama3.1-8B-{which_class}-sweeps')
    # sweep_id = "kc3muvie"
    def config_fn(config=None):
        return train_model(config=config, dataset=split_dataset, which_class=which_class)
    wandb.agent(sweep_id, config_fn, count=64)

# classifier_types = ['goodareas', 'badareas']
classifier_types = ['goodareas']
# SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
SKILL_OPTIONS = ["Reflections"]
for classifier_type in classifier_types:
    for skill in SKILL_OPTIONS:
        run_sweep(f"{skill}-{classifier_type}")

Create sweep with ID: rqxxgk4j
Sweep URL: https://wandb.ai/ryanlouie2021-stanford-university/modernbert-Reflections-goodareas-sweeps/sweeps/rqxxgk4j


wandb: Agent Starting Run: jnnpyy2y with config:
wandb: 	batch_size: 16
wandb: 	context_size: 1
wandb: 	downsampling_factor: 4
wandb: 	epochs: 4
wandb: 	learning_rate: 1.075826551350604e-06
wandb: 	lora_alpha: 16
wandb: 	lora_dropout: 0
wandb: 	lora_r: 64
wandb: 	weight_decay: 0
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 5343.00 examples/s]


{'conv_index': 358, 'helper_index': 2, 'input': ["Seeker: I'm okay, just a little stressed", 'Helper: is there anything specifically that is stressing you out the most right now?', "Seeker: I'm just having a hard time dealing with hurtful things people have said to me. I feel no self-worth.", "Helper: I'm really sorry to hear that you're feeling this way. It must be really difficult for you. Can you tell me more about these situations and how they made you feel?"], 'Reflections-goodareas': 0, 'Validation-goodareas': 1, 'Empathy-goodareas': 1, 'Questions-goodareas': 1, 'Suggestions-goodareas': 0, 'Self-disclosure-goodareas': 0, 'Structure-goodareas': 0, 'Professionalism-goodareas': 0, 'Reflections-badareas': 0, 'Validation-badareas': 0, 'Empathy-badareas': 0, 'Questions-badareas': 0, 'Suggestions-badareas': 0, 'Self-disclosure-badareas': 0, 'Structure-badareas': 0, 'Professionalism-badareas': 0, 'text': "Seeker: I'm just having a hard time dealing with hurtful things people have said to

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:06<00:00,  1.65s/it]
/tmp/rylouie/ipykernel_2998569/2333267081.py:313: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
Packing eval dataset: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 3087.62 examples/s]
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.

...

Run yeu4jgf9 errored:
Traceback (most recent call last):
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/wandb/agents/pyagent.py", line 306, in _run_job
    self._function()
  File "/tmp/rylouie/ipykernel_2998569/2010477411.py", line 5, in config_fn
    return train_model(config=config, dataset=split_dataset, which_class=which_class)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/rylouie/ipykernel_2998569/2333267081.py", line 221, in train_model
    model = AutoModelForCausalLM.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/models/auto/auto_factory.py", line 564, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/modeling_utils.py", line 262, in _wrapper
    return 

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 5203.72 examples/s]


{'conv_index': 358, 'helper_index': 2, 'input': ["Seeker: I'm okay, just a little stressed", 'Helper: is there anything specifically that is stressing you out the most right now?', "Seeker: I'm just having a hard time dealing with hurtful things people have said to me. I feel no self-worth.", "Helper: I'm really sorry to hear that you're feeling this way. It must be really difficult for you. Can you tell me more about these situations and how they made you feel?"], 'Reflections-goodareas': 0, 'Validation-goodareas': 1, 'Empathy-goodareas': 1, 'Questions-goodareas': 1, 'Suggestions-goodareas': 0, 'Self-disclosure-goodareas': 0, 'Structure-goodareas': 0, 'Professionalism-goodareas': 0, 'Reflections-badareas': 0, 'Validation-badareas': 0, 'Empathy-badareas': 0, 'Questions-badareas': 0, 'Suggestions-badareas': 0, 'Self-disclosure-badareas': 0, 'Structure-badareas': 0, 'Professionalism-badareas': 0, 'text': "Seeker: I'm just having a hard time dealing with hurtful things people have said to

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 818/818 [00:00<00:00, 3070.54 examples/s]
/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/accelerate/utils/modeling.py:1536: UserWarning: Current model requires 32.0 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Traceback (most recent call last):
  File "/tmp/rylouie/ipykernel_2998569/2333267081.py", line 221, in train_model
    model = AutoModelForCausalLM.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/models/auto/auto_factory.py", line 564, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/pyth

Run mjvrdxur errored:
Traceback (most recent call last):
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/wandb/agents/pyagent.py", line 306, in _run_job
    self._function()
  File "/tmp/rylouie/ipykernel_2998569/2010477411.py", line 5, in config_fn
    return train_model(config=config, dataset=split_dataset, which_class=which_class)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/rylouie/ipykernel_2998569/2333267081.py", line 221, in train_model
    model = AutoModelForCausalLM.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/models/auto/auto_factory.py", line 564, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/modeling_utils.py", line 262, in _wrapper
    return 

## Using the model to make predictions

In [186]:
import pandas as pd


condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"../Empathy-Mental-Health/dataset/all_{condition}_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c..."
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...


In [187]:
from transformers import pipeline
 
# load model from huggingface.co/models using our repository id
classifier = pipeline("sentiment-analysis", model=f"ModernBERT-{which_class}-classifier", device=0)
# classifier = pipeline("sentiment-analysis", model="ModernBERT-Empathy-goodareas-classifier", device=0)

# sample = f"Seeker: {input_data.loc[0, "seeker_post"]}\nHelper: {input_data.loc[0, "response_post"]}"
# pred = classifier(sample)
# print(pred)

def binary_prediction_seeker_response_post(seeker, helper):
    sample = f"Seeker: {seeker}\nHelper: {helper}"
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


In [188]:
# output_preds = input_data.apply(binary_prediction_seeker_response_post, axis=0)

strengths = [binary_prediction_seeker_response_post(input_data.loc[i, "seeker_post"], input_data.loc[i, "response_post"])
             for i in range(len(input_data))]

In [189]:
input_data[f"{which_class}"] = strengths
# input_data["Empathy-goodareas"] = strengths

In [193]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,Reflections-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,0
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,1
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,0
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,0


In [195]:
# input_data.to_csv(f'all_{condition}_seekerhelper_pairs_{which_class}.csv')
input_data.to_csv(f"N94_all_seekerhelper_pairs_{which_class}.csv")

In [192]:
f'all_{condition}_seekerhelper_pairs_{which_class}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'